In [26]:
import ollama
import chromadb

In [27]:
client = chromadb.PersistentClient(path="./chroma_db_optimized")
collection = client.get_or_create_collection(name="sales_data_optimized", metadata={"hnsw:space": "cosine"})
print(f"Loaded {collection.count()} chunks from ChromaDB")

Loaded 2257 chunks from ChromaDB


In [28]:
def retrieve_context(query, num_results=8):
    results = collection.query(query_texts=[query], n_results=num_results)
    return results['documents'][0] if results['documents'] else []

In [29]:
def rag_query(query, model="llama3.2:3b"):
    chunks = retrieve_context(query, num_results=8)
    if not chunks:
        return "No relevant data found."
    
    context = "\n\n".join(chunks)
    prompt = f"""You are a sales analytics expert. Answer based ONLY on the provided data.

Instructions:
- If data contains summary totals, prioritize those for your answer
- Provide specific numbers, not estimates
- Aggregate transaction data when needed to answer the question

Data:
{context}

Q: {query}
A:"""
    
    response = ollama.generate(model=model, prompt=prompt, stream=False)
    return response['response']

In [30]:
MODEL = "llama3.2:3b"

queries = {
    "TREND ANALYSIS": [
        "What is the sales trend over the 4-year period?",
        "Which months show the highest sales? Is there seasonality?",
        "How has profit margin changed over time?"
    ],
    "CATEGORY ANALYSIS": [
        "Which product category generates the most revenue?",
        "What sub-categories have the highest profit margins?",
        "Which products are frequently sold at a discount?"
    ],
    "REGIONAL ANALYSIS": [
        "Which region has the best sales performance?",
        "Compare sales performance across different states.",
        "Which cities are the top performers?"
    ],
    "COMPARATIVE ANALYSIS": [
        "Compare Technology vs Furniture sales trends.",
        "How does the West region compare to the East in terms of profit?"
    ]
}

for category, qs in queries.items():
    print(f"\n{category}")
    for q in qs:
        print(f"\nQ: {q}")
        answer = rag_query(q, model=MODEL)
        print(f"A: {answer}\n")


TREND ANALYSIS

Q: What is the sales trend over the 4-year period?
A: Based on the provided data, I can identify some general trends and observations about the sales of different product categories over a 4-year period:

1. Office Supplies - Art:
The sales of office supplies art products have been relatively consistent across the four years, with most months showing sales in the range of $2-$17 per unit.
However, there is an overall decline in sales of this category, as seen on August 16, 2015, where a Newell 346 product sold for $0.26, and on April 22, 2016, where a Barrel Sharpener product sold for $8.35.

2. Technology - Phones:
The sales of technology phones have shown some fluctuations across the four years, with most months showing sales in the range of $400-$1,300 per unit.
There is an overall increase in sales of this category, as seen on August 16, 2015, where Samsung Galaxy S4 Mini products sold for $211.50 each.

3. Furniture - Furnishings:
The sales of furniture furnishing